In [1]:
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import numpy as np
import optuna
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.dataset import TimeSeriesDataset
from src.eval.scorer import VolatilityScorer
from src.models.markov_tcn_wrapper import MarkovTCNModel, markov_tcn_search_space
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import test_toolkit as ts
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "markov_tcn"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"¡GPU Detectada! Usando: {torch.cuda.get_device_name(0)}")
else:
    print("ALERTA: No se detectó GPU")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)
ALERTA: No se detectó GPU


:::{admonition} Resumen
:class: info
- El modelo *Markov-TCN* se evaluó mediante un esquema de ventana expansiva con reentrenamiento en cada paso para el pronóstico de la volatilidad diaria del crudo Brent.
- En la comparación entre entrenamiento y prueba, las métricas de escala acotada se mantuvieron dentro del umbral de 0.05 (con QLIKE = $0.4026$ en prueba, sin superar al $0.3675$ de la `TCN`), mientras que HMSE ($\Delta_{\text{abs}} = +1.4801$) y HMAE ($\Delta_{\text{abs}} = +0.0677$) aumentaron fuera de muestra y los residuos mostraron colas pesadas mejor descritas por una distribución $t$ de Student ($\nu = 6.5685$) que por la normal.
- En evaluación fuera de muestra (*walk-forward*, penalizado por la estimación de Markov en CPU y las transferencias hacia la GPU), no se observó autocorrelación lineal ni dependencia no lineal en dimensión 2, aunque persistió un efecto ARCH significativo en el rezago 20 ($p_{\text{adj}} = .035$).
- En el periodo de prueba y el episodio de volatilidad de 2026, el modelo siguió la dinámica general con ligera tendencia superior en calma relativa, pero subestimó el pico principal de marzo ($\sigma_R \approx 0.25$ frente a $\hat{\sigma}_t \approx 0.10$) y reaccionó con desfase sobreestimando repuntes alrededor de abril ($\hat{\sigma}_t \approx 0.12$), ofreciendo un ajuste distribucional sólido pero sin superar a la `TCN` base y con un costo computacional elevado por optimizar.
:::

## Introducción

La Red Convolucional Temporal con Cambio de Régimen de Markov (*Markov-Switching Temporal Convolutional Network*, `MarkovTCNModel`) es una arquitectura híbrida diseñada para capturar dependencias temporales no lineales a lo largo de una ventana histórica de longitud $L$ (`look_back`) y cambios de régimen en la volatilidad mediante canales causales de probabilidad de estado. A diferencia de una TCN univariada, que procesa únicamente la serie de retornos estandarizados, este modelo incorpora probabilidades filtradas de estados ocultos estimadas de forma causal, permitiendo condicionar la dinámica convolucional al régimen latente sin introducir fuga de información futura.

En series financieras, esta formulación combina la estructura de regímenes de un modelo autorregresivo de Markov (*Markov-Switching Autoregression*) para distinguir estados de distinta varianza con la modelación convolucional causal dilatada para pronosticar la varianza futura del crudo Brent.

Esta sección presenta, entrena y evalúa `MarkovTCNModel` para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. Se examina el efecto de la arquitectura (`look_back`, `num_levels`, `channels`, `kernel_size`, `num_convs`), los parámetros del filtro de Markov (`ma_window`, `ar_order`, `k_regimes`) y la regularización (`dropout`, `weight_decay`) sobre el desempeño predictivo, bajo las mismas condiciones de información, pérdida QLIKE y evaluación *walk-forward* que el resto de modelos.

### Arquitectura

La arquitectura de `MarkovTCNModel` consta de dos etapas acopladas: un filtro causal de regímenes de Markov sobre la serie continua de retornos y una red convolucional temporal multicanal (`MarkovTCNNetwork`) formada por bloques residuales dilatados (`TCNBlock`) y una salida positiva mediante *Softplus*.

A partir de la serie univariada de retornos estandarizados $z_t$, se calcula un promedio móvil causal retrospectivo de longitud `ma_window` ($W_{\text{MA}}$):

$$
\bar{r}_t = \frac{1}{\min(t, W_{\text{MA}})} \sum_{j=0}^{\min(t, W_{\text{MA}})-1} z_{t-j},
$$

y se estandariza sobre el segmento de ajuste hasta $T_{\text{fit}}$ (`fit_end_idx`). Sobre esa serie se estima por máxima verosimilitud un proceso autorregresivo de orden $p$ (`ar_order`) con $K$ regímenes (`k_regimes`) y varianza conmutada (`switching_variance=True`, `switching_ar=False`):

$$
\tilde{r}_t = c_{s_t} + \sum_{m=1}^{p} \phi_m \tilde{r}_{t-m} + \sigma_{s_t} \epsilon_t, \qquad s_t \in \{0, \dots, K-1\}, \quad \epsilon_t \sim \mathcal{N}(0, 1),
$$

donde $s_t$ sigue una cadena de Markov de primer orden con matriz de transición $\mathbf{P}$. La estimación emplea una escala progresiva (`scale_ladder`) y un respaldo causal logístico (`_causal_fallback_probs`) ante fallos de convergencia. Tras ordenar los regímenes según su varianza condicional empírica ascendente dentro de la muestra de ajuste, el filtrado hacia adelante (*forward filtering*) produce las probabilidades causales:

$$
p_{t,k} = \mathbb{P}(s_t = k \mid \mathcal{F}_t), \qquad k \in \{1, \dots, K-1\},
$$

omitiendo el régimen $k = 0$ por redundancia ($\sum_{k=0}^{K-1} p_{t,k} = 1$). Cada instante queda representado por el vector multicanal de dimensión $F = 1 + \max(0, K - 1)$:

$$
\mathbf{x}_t = \left[ z_t, \, p_{t,1}, \, \dots, \, p_{t,K-1} \right]^{\top} \in \mathbb{R}^{F},
$$

organizado en ventanas históricas $\mathbf{X}_t \in \mathbb{R}^{L \times F}$.

Cada ventana $\mathbf{X}_t$ se transpone a formato canal-primero $\mathbf{H}^{(0)} \in \mathbb{R}^{F \times L}$ y pasa por $B = \text{num\_levels}$ bloques residuales (`TCNBlock`) con dilatación $d_i = 2^i$ para $i \in \{0, \dots, B-1\}$.

Cada bloque utiliza convoluciones unidimensionales con relleno izquierdo (`CausalConv1d`) de tamaño $(k - 1) \cdot d_i$, donde $k$ es `kernel_size`, encadenando $M = \text{num\_convs}$ capas de convolución causal, ReLU y *Dropout*:

$$
\mathbf{U}^{(i,m)} = \text{Dropout}\left(\text{ReLU}\left(\text{CausalConv1d}_{d_i}\left(\mathbf{U}^{(i,m-1)}\right)\right)\right), \qquad m = 1, \dots, M,
$$

con $\mathbf{U}^{(i,0)} = \mathbf{H}^{(i)}$, y conexión residual con proyección $1 \times 1$ ($\mathbf{W}_{\text{res}}^{(i)}$) cuando cambia el número de canales hacia el ancho oculto $C$ (`channels`):

$$
\mathbf{H}^{(i+1)} = \text{ReLU}\left(\mathbf{U}^{(i,M)} + \mathcal{R}\left(\mathbf{H}^{(i)}\right)\right),
$$

donde $\mathcal{R}(\mathbf{H}^{(i)}) = \mathbf{H}^{(i)}$ si el número de canales se mantiene o $\mathbf{W}_{\text{res}}^{(i)} * \mathbf{H}^{(i)}$ en caso contrario. En la búsqueda con Optuna, el mínimo de niveles se define como:

$$
B_{\min} = \left\lceil \log_2\left(1 + \frac{L - 1}{M (k - 1)}\right) \right\rceil,
$$

para asegurar cobertura completa de la ventana $L$.

Sobre el último paso temporal del último bloque residual, $\mathbf{h}_L^{(B)} \in \mathbb{R}^{C}$, una capa lineal y la función *Softplus* con piso $\varepsilon = 10^{-8}$ generan la varianza pronosticada positiva:

$$
y_t = \mathbf{w}_{\text{fc}}^{\top}\mathbf{h}_L^{(B)} + b_{\text{fc}}, \qquad \widehat{RS}_t = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ reúne los parámetros entrenables de la red.

Bajo el protocolo del *benchmark*, los retornos logarítmicos $r_t$ se estandarizan mediante Z-score ajustado exclusivamente en el segmento de entrenamiento de cada *fold*, y la varianza realizada de Rogers-Satchell se escala con `target_scale`. El modelo realiza predicción a un paso adelante heredando de `BasePyTorchVolatilityModel` la evaluación *walk-forward* (`retrain_freq=1`), optimización Adam, *weight decay*, *Dropout* y parada temprana (`patience=20`, `max_epochs=200`) bajo la pérdida **QLIKE**:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de ventanas de entrenamiento y $RS_i$ es la varianza de Rogers-Satchell objetivo. La implementación utiliza `PyTorch` (`2.13.0`) y `statsmodels`.


## Entrenamiento

Para la selección de hiperparámetros del modelo `MarkovTCNModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los demás modelos del *benchmark*. A diferencia de las arquitecturas sin modelado latente de regímenes, la búsqueda con Optuna exploró conjuntamente la dinámica de cambio de régimen de Markov, el bloque convolucional dilatado y los parámetros de regularización.

Antes de construir las ventanas deslizantes de cada partición del esquema *Walk-Forward* (`n_splits=5`, `embargo=1`, y `warmup` definido según la ejecución, con `exec_id = 7` y `current_warmup = 1270`), se estimaron y filtraron causalmente las probabilidades de régimen sobre la serie continua de retornos hasta el punto de corte correspondiente para incorporarlas como canales causales junto con los retornos estandarizados. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de la matriz de rezagos (*lag matrix*) contenía una secuencia histórica multicanal de longitud `look_back`, utilizada como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque preservó el orden temporal de las observaciones y evitó cualquier fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento periódico (`retrain_freq=1`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el tamaño del núcleo convolucional (`kernel_size`), el número de convoluciones por bloque (`num_convs`) y el número de niveles dilatados (`num_levels`), acotado entre un mínimo calculado según la cobertura temporal requerida y un máximo superior. Asimismo, se exploraron el ancho de canales (`channels`), la tasa de abandono (`dropout`), la tasa de aprendizaje (`learning_rate`), la penalización $\ell_2$ (`weight_decay`), la ventana de media móvil causal (`ma_window`), el orden autorregresivo (`ar_order`) y el número de regímenes (`k_regimes`), fijando `seed = 42`.

El entrenamiento de cada red se configuró con un máximo de 200 épocas (`max_epochs = 200`) y una paciencia de 20 épocas (`patience = 20`) para la parada temprana interna, bajo un esquema de búsqueda con `n_trials = 50` y `early_stop_rounds = 15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def markov_tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = math.ceil(
        math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "num_levels": trial.suggest_int("num_levels", min_levels, max_levels),
        "channels": trial.suggest_categorical("channels", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "ma_window": trial.suggest_categorical("ma_window", [5, 10, 22]),
        "ar_order": trial.suggest_int("ar_order", 1, 3),
        "k_regimes": trial.suggest_categorical("k_regimes", [2, 3]),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 7
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=MarkovTCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=markov_tcn_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

Aunque el protocolo experimental contemplaba un presupuesto de 50 ensayos (`n_trials=50`), el estudio de Optuna registró 17 ensayos en esta ejecución (con 13 de 47 iteraciones restantes en la barra de progreso tras unas 17 horas y 40 minutos), completando 14 ensayos válidos (82.4 %), con 1 ensayo podado y 0 fallidos, y alcanzando un mejor valor de QLIKE de 0.4780 (en el ensayo 3). La poda temprana descartó configuraciones poco competitivas durante la búsqueda, mientras que la arquitectura con canales causales de régimen de Markov y bloques convolucionales dilatados estabilizó su entrenamiento mediante conexiones residuales y regularización para completar las evaluaciones sin fallos.

El avance de la búsqueda en Optuna acumuló unas 17 horas y 40 minutos hasta el punto reportado, mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó unos 9 minutos, con un promedio de 7.0580 segundos por partición. Este tiempo refleja el costo computacional de estimar y filtrar causalmente la cadena de Markov junto con el entrenamiento de redes convolucionales dilatadas y su reentrenamiento periódico (`retrain_freq=1`), conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [3]:
resumen_data = []

exec_id = 7
model_tag = "MarkovTCNModel"
study_name_override = None
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Exec": exec_id,
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
7,17,14,1,0,0.4780,536.4400,7.0580


El ranking de las cinco mejores configuraciones de `MarkovTCNModel` mostró un consenso mayoritario en el uso de ventanas de 28 días (`Lookback = 28` en cuatro de las cinco posiciones) y una media móvil causal corta de 5 días (`MA Win = 5` en cuatro posiciones), con `Canales = 64` en las cuatro primeras configuraciones y `Kernel = 2` en todas las posiciones. Esto sugiere que el modelo híbrido favorece una ventana de contexto más amplia para combinar la señal convolucional con las probabilidades causales de régimen, mientras que frente a la `TCN` estándar (cuyo mejor promedio es `0.437 ± 0.111` con `Lookback = 14`, `Kernel = 3` y `5` niveles) la TCN mantiene un menor QLIKE medio de validación y menor dispersión entre pliegues.

In [4]:
csv_path = exec_dir / "MarkovTCNModel_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("*_results.csv"))
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_mtcn = pd.read_csv(csv_path)

df_mtcn["lr"] = df_mtcn["learning_rate"].apply(lambda x: f"{x:.4f}")
df_mtcn["wd"] = df_mtcn["weight_decay"].apply(lambda x: f"{x:.1e}")
df_mtcn["drop_fmt"] = df_mtcn["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back",
    "ma_window",
    "ar_order",
    "k_regimes",
    "num_levels",
    "channels",
    "kernel_size",
    "num_convs",
    "drop_fmt",
    "lr",
    "wd",
]

df_agg = (
    df_mtcn.groupby(cols_agrupar, as_index=False)["qlike"]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_top5 = df_agg.sort_values(by="mean", ascending=True).head(5).rename(
    columns={
        "look_back": "Lookback",
        "ma_window": "MA Win",
        "ar_order": "AR Order",
        "k_regimes": "Regímenes",
        "num_levels": "Niveles TCN",
        "channels": "Canales",
        "kernel_size": "Kernel",
        "num_convs": "Convs",
        "drop_fmt": "Dropout",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

cols_finales = [
    "Lookback", "MA Win", "AR Order",
    "Regímenes", "Niveles TCN", "Canales",
    "Kernel", "Convs", "Dropout", "LR",
    "Weight Decay", "QLIKE"
]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,MA Win,AR Order,Regímenes,Niveles TCN,Canales,Kernel,Convs,Dropout,LR,Weight Decay,QLIKE
28,5,3,2,4,64,2,3,0.287,0.0075,9.3e-04,0.478 ± 0.206
28,5,2,3,5,64,2,3,0.125,0.0084,3.3e-04,0.486 ± 0.198
28,5,2,2,5,64,2,1,0.297,0.0029,6.4e-04,0.527 ± 0.288
28,22,2,2,5,64,2,3,0.050,0.0064,7.4e-04,0.529 ± 0.285
7,5,3,3,3,32,2,1,0.289,0.0070,1.4e-05,0.578 ± 0.439


Entre las dos primeras configuraciones de `MarkovTCNModel`, el primer lugar obtuvo una menor pérdida media de validación (`0.478` frente a `0.486`) con `2` regímenes, `AR Order = 3`, `4` niveles TCN y `Dropout = 0.287`, frente a `3` regímenes, `AR Order = 2`, `5` niveles TCN y `Dropout = 0.125` en el segundo lugar. La elección de la primera configuración responde a su menor QLIKE medio de validación dentro del estudio, mientras que a partir de la tercera posición tanto la pérdida media como la dispersión entre *folds* aumentan de forma marcada respecto a la `TCN` base.

In [ ]:
best_markov_tcn = MarkovTCNModel(
    look_back=28,
    ma_window=5,
    ar_order=3,
    k_regimes=2,
    num_levels=4,
    channels=64,
    kernel_size=2,
    num_convs=3,
    dropout=0.287,
    learning_rate=0.0075,
    weight_decay=9.3e-04,
    max_epochs=200,
    patience=20,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_markov_tcn.fit(y_train, rs_train)
fit_elapsed = time.time() - start_time

El entrenamiento del `MarkovTCNModel` sobre toda la muestra tomó 16.03 segundos. En la curva de aprendizaje, tras valores iniciales superiores a 6.0 en entrenamiento y cercanos a 2.8 en validación junto con oscilaciones en las primeras épocas, ambas trayectorias descendieron y se estabilizaron en valores bajos de QLIKE, fijando los mejores pesos en la época 23 y continuando hasta la época 43 sin brechas crecientes en validación que indicaran sobreajuste.

Frente a este comportamiento, la `TCN` completó el entrenamiento en 5.05 segundos y partió de un valor inicial menor en validación (cerca de 1.2), aunque en ambos modelos el entrenamiento descendió desde encima del eje superior en las primeras épocas con validación por debajo en el tramo inicial. La `TCN` seleccionó los mejores pesos en la época 20 y finalizó en la época 40, mientras que el `MarkovTCNModel` presentó menor amplitud de variación entre épocas tras la época 30.

In [7]:
import pandas as pd

history_csv_path = exec_dir / "markov_tcn_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

n_epochs = len(history_df)
x_max = max(n_epochs + 1, 2)
x_step = max(1, round(x_max / 11))

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0, title_size=14,
    label_size=14, tick_size=12,
    legend_size=14, ylim=(0, 6),
    xlim=(0, x_max), y_step=0.5,
    x_step=x_step, legend_bbox=(0.95, 0.95)
)

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no fue posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit`, se calcularon los canales de probabilidad causal de régimen mediante `_augment_series` con el estado de Markov estimado (`k_regimes = 2`, `ma_window = 5`, `ar_order = 3`) y se escaló el objetivo de varianza, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 28 días (`look_back = 28`) y finalmente se evaluó en modo `eval()` sin gradientes. Las varianzas estimadas se devolvieron a su escala original y se alinearon con los datos observados descartando las observaciones iniciales requeridas por la ventana.

In [ ]:
best_markov_tcn.net.eval()
look_back = best_markov_tcn.context_length
target_scale = getattr(best_markov_tcn, "target_scale", 1.0)

y_train_scaled = best_markov_tcn.scaler.transform(
    np.asarray(y_train).reshape(-1, 1)
).flatten()
y_train_aug = best_markov_tcn._augment_series(y_train_scaled, fit_end_idx=None)
rs_train_scaled = np.asarray(rs_train) * target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_aug,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=getattr(best_markov_tcn, "batch_size", 128),
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_markov_tcn.device)
        preds_batch = (
            best_markov_tcn.net(x_batch).reshape(-1).detach().cpu().numpy()
        )
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
y_train_aligned = np.asarray(y_train)[start_idx : start_idx + n_preds]
rs_train_aligned = np.asarray(rs_train)[start_idx : start_idx + n_preds]

df_train_preds = pd.DataFrame(
    {
        "y_true": y_train_aligned,
        "rs_true": rs_train_aligned,
        "y_pred": preds_train_var,
    },
    index=dates_aligned,
)

train_output_path = results_dir / "markov_tcn_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=True)

A partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y de los retornos observados ($y_t$), se evaluó el ajuste del modelo sobre el conjunto de entrenamiento mediante los residuos estandarizados:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .278$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .369$). Las pruebas ARCH-LM no rechazaron la hipótesis nula de homocedasticidad condicional en ninguno de los tres rezagos evaluados ($p_{\text{adj}} \ge .415$), lo que indicó que la varianza condicional estimada absorbió el agrupamiento de volatilidad dentro de la muestra de entrenamiento. Las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), reflejando colas pesadas en los retornos del crudo Brent.

In [8]:
train_output_path = exec_dir / "markov_tcn_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

n_preds = len(preds_train_var)
y_arr = np.asarray(y_train).reshape(-1)
y_train_aligned = y_arr[-n_preds:]

std_residuals_train = y_train_aligned / np.sqrt(np.maximum(preds_train_var, 1e-12))
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),8.9785,.534,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),27.8048,.114,.342,Sin Autocorrelación
Ljung-Box (Lag 30),40.6586,.093,.278,Sin Autocorrelación
ARCH-LM (Lag 10),9.6646,.470,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),20.6927,.415,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),28.0805,.566,1.000,Sin Efectos ARCH
Jarque-Bera,1061.4298,< .001,--,Residuos No Normales
Lilliefors,0.0551,< .001,--,Residuos No Normales
BDS (Dim 2),-0.8982,.369,--,Sin Estructura No Lineal


Los residuos estandarizados del modelo mostraron un ajuste a una distribución $t$ de Student ($\nu = 4.6608$). En los cuantiles extremos de ambas colas se observaron desviaciones respecto a la referencia teórica, con los cuantiles superiores empíricos situándose por debajo de los teóricos y algunos cuantiles inferiores ligeramente por debajo de la diagonal. Estas discrepancias en las colas implicaron que la aproximación $t$ de Student sobrestimó la magnitud de los eventos positivos extremos en comparación con la cola empírica de los residuos, reflejando un ajuste más preciso en la región central que en los valores atípicos. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .4120$; Cramér-von Mises: $p = .3671$) no rechazaron la hipótesis nula de ajuste a la distribución $t$.

In [10]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.8),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal ($p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 195.72$ y $\Delta\text{BIC} = 190.05$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara las colas pesadas de los residuos estandarizados del modelo en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [11]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-2,958.17","5,920.33","5,931.67",-,-
t-Student,"-2,859.31","5,724.61","5,741.62",197.72,< .001


Las predicciones de volatilidad del `MarkovTCNModel` $\left(\hat{\sigma}_t\right)$ siguieron la trayectoria de la volatilidad realizada $\left(\sigma_R\right)$ a lo largo de todo el periodo, capturando tanto los episodios de calma como los agrupamientos de alta turbulencia. No se observó subestimación ni sobreestimación sistemática de la volatilidad, aunque durante los tramos de baja volatilidad (como en 2014 y entre 2017 y 2018) la serie estimada mantuvo un nivel base ligeramente superior al de la volatilidad realizada, mientras que durante la crisis de COVID-19 (marzo y abril de 2020) respondió con incrementos abruptos que sobreestimaron puntualmente los dos picos principales frente a $\sigma_R$.

En comparación con la `TCN` y el `Causal Transformer`, el `MarkovTCNModel` presentó una respuesta más pronunciada ante los choques agudos de 2020 y mantuvo un nivel base de volatilidad similar al del `Causal Transformer` y algo superior al de la `TCN` en los periodos tranquilos. Frente a la `TCN`, cuya estimación tendió a mantenerse por debajo de los picos extremos y más atenuada durante los episodios de tensión, el `MarkovTCNModel` alcanzó mayores amplitudes en los máximos de 2020 y una respuesta más marcada en el episodio de 2022. Respecto al `Causal Transformer`, ambas trayectorias mostraron un seguimiento cercano a lo largo de toda la serie y niveles elevados en los picos extremos, con el `MarkovTCNModel` registrando un mayor impulso en el primer pico de 2020 y una respuesta menor en el repunte de 2022.

In [13]:
look_back = 28
n_preds = len(preds_train_var)
start_idx = len(y_train) - n_preds

dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(np.asarray(rs_train)[start_idx : start_idx + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Markov-TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Markov-TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental a partir de la diferencia entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas de escala acotada (como QLIKE, CRPS y pérdidas *pinball*), de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra, mientras que en métricas de error relativo como HMSE y HMAE se examina la magnitud del salto en relación con su escala y los episodios de baja volatilidad.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo alcista o defensivo puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra se generaron mediante el método `forecast` del modelo, aplicando una ventana expansiva con reentrenamiento en cada paso (`retrain_freq = 1`) sin fuga de información y devolviendo las varianzas condicionales en su escala original para compararlas con la varianza realizada.

In [ ]:
start_time = time.time()
preds_test = best_markov_tcn.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

print(f"Tiempo de pronóstico: {preds_time / 60:.2f} minutos ({preds_time:.2f} s)")

preds_test_arr = np.asarray(preds_test).flatten()
rs_test_arr = np.asarray(rs_test).flatten()
y_test_arr = np.asarray(y_test).flatten()

n_test = len(preds_test_arr)
dates_test_aligned = df_test.index[-n_test:]
rs_test_aligned = rs_test_arr[-n_test:]
y_test_aligned = y_test_arr[-n_test:]

df_preds = pd.DataFrame(
    {
        "date": dates_test_aligned,
        "y_true": y_test_aligned,
        "rs_true": rs_test_aligned,
        "y_pred": preds_test_arr,
    }
)

output_path = exec_dir / "markov_tcn_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

print(f"Guardado en: {output_path}")

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento en cada paso tuvo una duración total de 207.92 minutos (12475.20 s) para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio de 0.22 minutos (unos 13.09 segundos) por observación. Este tiempo es sustancialmente superior no solo al de los modelos paramétricos, cuyo cómputo toma milisegundos, sino también al de la `TCN` y el `Causal Transformer`, reflejando el costo de estimar las probabilidades de régimen de Markov en CPU, transferir los datos hacia la GPU y reajustar los pesos de la red en cada paso de la ventana de evaluación.

El diagnóstico de los residuos estandarizados fuera de muestra mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 $\left(p_{\text{adj}} \ge .088\right)$, así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 $\left(p = .061\right)$. En las pruebas ARCH-LM, el rezago 20 presentó efectos ARCH significativos tras el ajuste de Bonferroni $\left(p_{\text{adj}} = .035\right)$, mientras que en los rezagos 10 y 30 no se rechazó la hipótesis nula de homocedasticidad condicional ($p_{\text{adj}} = .164$ y $p_{\text{adj}} = .235$, respectivamente). Asimismo, las pruebas de Jarque-Bera $\left(p < .001\right)$ y Lilliefors $\left(p < .001\right)$ rechazaron la normalidad de los residuos.

In [16]:
preds_path = exec_dir / "markov_tcn_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].to_numpy()

n_preds = len(preds_test_var)
y_test_arr = np.asarray(y_test).reshape(-1)
y_test_aligned = y_test_arr[-n_preds:]

std_residuals_oos = y_test_aligned / np.sqrt(np.maximum(preds_test_var, 1e-12))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),11.2277,.340,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),33.5715,.029,.088,Sin Autocorrelación
Ljung-Box (Lag 30),39.4541,.116,.348,Sin Autocorrelación
ARCH-LM (Lag 10),18.0096,.055,.164,Sin Efectos ARCH
ARCH-LM (Lag 20),37.0368,.012,.035,Efectos ARCH Significativos
ARCH-LM (Lag 30),41.5300,.078,.235,Sin Efectos ARCH
Jarque-Bera,124.6018,< .001,--,Residuos No Normales
Lilliefors,0.0446,< .001,--,Residuos No Normales
BDS (Dim 2),-1.8704,.061,--,Sin Estructura No Lineal


La inspección visual de los residuos estandarizados fuera de muestra muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 6.5685$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica a lo largo del centro y gran parte de los extremos, observándose únicamente leves desviaciones por debajo de la línea de referencia en los cuantiles inferiores (alrededor de $-4$ a $-2$) y ligeras dispersiones en los cuantiles superiores alrededor de $2$ a $4$, mientras que los extremos más altos vuelven a alinearse con la diagonal. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .6832$; Cramér-von Mises: $p = .6776$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [18]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal en los residuos estandarizados en el conjunto de prueba ($\text{LRT} = 44.14$, $p < .001$). Los criterios de información mostraron una reducción de $\Delta\text{AIC} = 42.14$ y $\Delta\text{BIC} = 37.28$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados fuera de muestra en comparación con el ajuste Gaussiano.

In [20]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,385.53","2,775.06","2,784.78",-,-
t-Student,"-1,363.46","2,732.92","2,747.50",44.14,< .001


En la comparación entre entrenamiento y prueba, las métricas de escala acotada (QLIKE, CRPS, pérdidas *pinball*, MSE, RMSE y MAE) se mantuvieron dentro del umbral de $|\Delta_{\text{abs}}| = 0.05$, mientras que HMSE ($\Delta_{\text{abs}} = +1.4801$) y HMAE ($\Delta_{\text{abs}} = +0.0677$) registraron incrementos fuera de muestra. Estas diferencias no implican necesariamente sobreajuste: pueden responder a cambios en el régimen de volatilidad o al pronóstico secuencial frente al ajuste *in-sample*. En particular, la ligera caída de QLIKE en prueba se asocia a la asimetría de la pérdida y al régimen evaluado, y el alza de HMSE y HMAE refleja la sensibilidad de los errores heterocedásticos relativos en escenarios de baja varianza o choques puntuales.

In [17]:
look_back = 28
n_preds_train = len(preds_train_var)
rs_train_arr = np.asarray(rs_train).reshape(-1)
rs_train_aligned = rs_train_arr[-n_preds_train:]

n_preds_test = len(preds_test_var)
rs_test_arr = np.asarray(rs_test).reshape(-1)
rs_test_aligned = rs_test_arr[-n_preds_test:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test_aligned,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4181,0.4026,-0.0156
CRPS,0.0127,0.0123,-0.0004
Pinball (τ=0.05),0.0030,0.0027,-0.0003
Pinball (τ=0.95),0.0014,0.0016,+0.0002
MSE,0.0000,0.0000,-0.0000
RMSE,0.0034,0.0022,-0.0012
MAE,0.0005,0.0004,-0.0001
HMSE,0.8802,2.3603,+1.4801
HMAE,0.6227,0.6904,+0.0677


La inspección visual de las predicciones fuera de muestra del modelo *Markov-TCN* evidenció que capturó la dinámica general de la volatilidad realizada ($\sigma_R$) y tendió a ubicarse ligeramente por encima en los periodos de calma relativa. Sin embargo, ante el choque de volatilidad de 2026, subestimó la magnitud del pico máximo alcanzado en marzo ($\sigma_R \approx 0.25$ frente a $\hat{\sigma}_t \approx 0.10$) y reaccionó con cierto desfase, llegando a sobreestimar repuntes secundarios posteriores alrededor de abril ($\hat{\sigma}_t \approx 0.12$). Este comportamiento, que contrasta con su ajuste cercano sobre los picos del conjunto de entrenamiento, ayudó a explicar el incremento de HMSE ($\Delta_{\text{abs}} = +1.4801$) y la persistencia de heterocedasticidad residual en el rezago 20.

In [22]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(np.maximum(df_preds_test["rs_true"], 0.0))
pred_test_vol = np.sqrt(np.maximum(df_preds_test["y_pred"], 0.0))

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (Markov-TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Markov-TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del modelo *Markov-TCN* permitieron contrastar el desempeño de una arquitectura convolucional temporal acoplada a un esquema de regímenes de Markov para el pronóstico de la volatilidad diaria del crudo Brent. En la comparación entre entrenamiento y prueba, las métricas de escala acotada (como QLIKE, CRPS, pérdidas *pinball*, MSE, RMSE y MAE) se mantuvieron dentro del margen de $|\Delta_{\text{abs}}| = 0.05$, mientras que HMSE ($\Delta_{\text{abs}} = +1.4801$) y HMAE ($\Delta_{\text{abs}} = +0.0677$) aumentaron fuera de muestra sin que ello implique necesariamente sobreajuste, dado que puede asociarse a cambios de régimen o a la evaluación secuencial frente al ajuste *in-sample*. En términos comparativos, el modelo no supera a la `TCN` base en QLIKE ($0.4026$ frente a $0.3675$), por lo que no mejora el desempeño predictivo de la referencia convolucional pura.

En el diagnóstico fuera de muestra, los residuos estandarizados no mostraron autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .088$) ni dependencia no lineal en la prueba BDS en dimensión 2 ($p = .061$). En las pruebas ARCH-LM, se detectó un efecto ARCH significativo en el rezago 20 ($p_{\text{adj}} = .035$), pero no en los rezagos 10 ($p_{\text{adj}} = .164$) ni 30 ($p_{\text{adj}} = .235$), señalando heterocedasticidad residual puntual en ese horizonte. Asimismo, se rechazó la normalidad en Jarque-Bera ($p < .001$) y Lilliefors ($p < .001$), y los residuos presentaron colas pesadas bien descritas por una distribución $t$ de Student ($\nu = 6.5685$), con pruebas de Kolmogorov-Smirnov ($p = .6832$) y Cramér-von Mises ($p = .6776$) favorables a este ajuste.

En términos de trayectoria y cómputo, el modelo capturó la dinámica general de la volatilidad realizada, pero subestimó el pico de marzo de 2026 y mostró desfase con sobreestimación en repuntes posteriores alrededor de abril, contrastando con su ajuste más cercano sobre los picos del entrenamiento. Por su parte, el esquema de ventana expansiva con reentrenamiento en cada paso requirió 207.92 minutos (12,475.20 s) para las 953 observaciones de prueba (0.22 minutos o 13.09 segundos por observación en promedio), superando no solo a los modelos paramétricos sino también a la `TCN` y al *Causal Transformer* debido a la estimación de Markov en CPU, las transferencias entre CPU y GPU y el reajuste continuo de pesos. En conjunto, el *Markov-TCN* ofrece un modelado distribucional de colas pesadas consistente, pero no supera a la `TCN` en QLIKE y presenta limitaciones ante choques abruptos y un costo computacional elevado que conviene optimizar.